In [ ]:
%%html
<!-- CSS settings for this notebook -->
<style>
    h1 {color:purple}
    h2 {color:purple}
    h3 {color:#0099ff}
    hr {    
        border: 0;
        height: 3px;
        background: #333;
        background-image: linear-gradient(to right, ForestGreen, DeepSkyBlue, ForestGreen);
    }
</style>

In [ ]:
# enable high-res images in notebook 
%config InlineBackend.figure_format = 'retina'
#%matplotlib inline

# 16. Big Data: Hadoop, Spark, NoSQL and IoT 

# 16.1 Introduction
### Big Data
* Previous data-science case studies all focused on AI
* Here, we focus on the **big-data infrastructure that supports AI solutions**
* As **data grows exponentially**, we want to **learn** from that data&mdash;and at **blazing speed**
* Done with **sophisticated algorithms**, hardware, software and networking designs
* With **big data**, **machine learning** and **deep learning** can be even **more effective**

<hr style="height:2px; border:none; color:#000; background-color:#000;">

# 16.3 NoSQL Big-Data Databases (1 of 2)
* **Relational databases** store data in rectangular **tables**
* **Not efficient** as **data volume** and the numbers of **tables** and **relationships between them** increases
* Most data produced today is 
    * **Unstructured**&mdash;**photos**, **videos** and **natural language** (social-media posts, texts, ...), or
    * **Semi-structured**&mdash;**JSON** and **XML** documents
* **Metadata** adds structure to **unstructured data**, making it **semi-structured**
    * **Tweets** (as you saw earlier)
    * **YouTube videos**&mdash;**who posted** and **when**, **title**, **description**, ...

<hr style="height:2px; border:none; color:#000; background-color:#000;">

# 16.3 NoSQL Big-Data Databases (2 of 2)
* **NoSQL databases** are designed for 
    * **unstructured** and **semi-structured big-data** 
    * big data **storage and processing demands**
* **Big data** requires **massive databases**, which can be spread across data centers worldwide in huge **clusters** of commodity computers
* The name **NoSQL** originally meant what its name implies
* With **growing use of SQL in big data**—such as **SQL on Hadoop** and **Spark SQL**—now it's said to stand for **“Not Only SQL”** 

<hr style="height:2px; border:none; color:#000; background-color:#000;">

### Four Major Types of **NoSQL Databases**
* **key–value**
* **document**
* **columnar**
* **graph**
* Our NoSQL case study uses **MongoDB document database** &mdash; the most popular NoSQL database
* **Overviews** of the **NoSQL database types** 
    * [**Python Fundamentals LiveLessons videos**](https://learning.oreilly.com/videos/python-fundamentals/9780135917411/9780135917411-PFLL_Lesson16_20) 
    * [**Python for Programmers, Section 16.3**](https://learning.oreilly.com/library/view/python-for-programmers/9780135231364/ch16.xhtml#ch16lev1sec3)

<hr style="height:2px; border:none; color:#000; background-color:#000;">

# 16.4 Case Study: A MongoDB JSON Document Database 
* Store and search **JSON** for **streamed tweets** about **100 U.S. senators**
* Summarize **top 10** by **tweet count**
* Display **interactive map** containing **tweet count summaries**
* I **pre-executed this example** because we stream 10,000 tweets, which can take substantial time 
* **Possible enhancement** &mdash; Use **sentiment analysis** to count **positive**, **negative** and **neutral tweets** mentioning each senator’s **handle**

<hr style="height:2px; border:none; color:#000; background-color:#000;">

### Free Cloud-Based MongoDB Atlas Cluster 
* Requires **no installation** 
* Store up to **512MB of data**
* Can store more with
    * [**Free MongoDB Community Server**](https://www.mongodb.com/download-center/community), or 
    * **Paid MongoDB Atlas account**
* **Creating your MongoDB Atlas cluster**
    * I discuss the details of **signing up** for a MongoDB account, **creating the MongoDB Atlas Cluster**, **configuring** it and getting your **connection string** in my [**Python Fundamentals LiveLessons videos**](https://learning.oreilly.com/videos/python-fundamentals/9780135917411/9780135917411-PFLL_Lesson16_22https://learning.oreilly.com/videos/python-fundamentals/9780135917411) and in [**Python for Programmers, Section 16.4.1**](https://learning.oreilly.com/library/view/Python+for+Programmers,+First+Edition/9780135231364/ch16.xhtml#ch16lev2sec14)

<hr style="height:2px; border:none; color:#000; background-color:#000;">

### Python Libraries Required for Interacting with MongoDB
```
conda install -c conda-forge pymongo
conda install -c conda-forge dnspython
```
* **`pymongo` library** &mdash; interact with **MongoDB databases** from Python
* **`dnspython` library** &mdash; used as part of connecting to a **MongoDB Atlas Cluster**

<hr style="height:2px; border:none; color:#000; background-color:#000;">

### keys.py 
* **`keys.py`** must contain 
    * your **Twitter credentials** 
    * Your **MongoDB connection string** 

<hr style="height:2px; border:none; color:#000; background-color:#000;">

## 16.4.2 Processing Tweets Stored in MongoDB

In [ ]:
import keys

<hr style="height:2px; border:none; color:#000; background-color:#000;">

### Loading the Senators’ Data (1 of 2)
* **`senators.csv`** (provided in notebook's folder) contains each senator's 
    * two-letter state code
    * name
    * party 
    * Twitter handle
    * Twitter ID
* **Twitter handle and ID** used to track tweets **to**, **from** and **mentioning** each senator 
* When following users via **numeric Twitter IDs**, must submit IDs as **strings**

<hr style="height:2px; border:none; color:#000; background-color:#000;">

### Loading the Senators’ Data (2 of 2)

In [ ]:
import pandas as pd

In [ ]:
senators_df = pd.read_csv('senators.csv')

In [ ]:
senators_df.head()

<hr style="height:2px; border:none; color:#000; background-color:#000;">

### Configuring the `pymongo` `MongoClient` 

In [ ]:
from pymongo import MongoClient

In [ ]:
atlas_client = MongoClient(keys.mongo_connection_string)

<hr style="height:2px; border:none; color:#000; background-color:#000;">

### Get **`pymongo` `Database`** Object Representing the `senators` Database
* **Creates the database** if it does not exist
* Will be used to store the collection of **tweet JSON documents**

In [ ]:
db = atlas_client.senators 

<hr style="height:2px; border:none; color:#000; background-color:#000;">

### Counting Tweets for Each Senator (1 of 2)
* MongoDB **text search** requires a **text index** specifying **document field(s) to search** 
	* MongoDB [index types](https://docs.mongodb.com/manual/indexes), [text indexes](https://docs.mongodb.com/manual/core/index-text) and [operators](https://docs.mongodb.com/manual/reference/operator)
* A **text index** is defined as a **tuple** containing **field name** to search and **index type** (`'text'`)
* **Wildcard field name (\$\*\*)** indexes **all** text fields for a **full-text search**

In [ ]:
db.tweets.create_index([('$**', 'text')])

<hr style="height:2px; border:none; color:#000; background-color:#000;">

### Counting Tweets for Each Senator (2 of 2)
* Use **`tweets` `Collection`’s `count_documents` method** and **full-text search** to count the total number of documents in the collection that contain the specified text
    * Find every **twitter handle** in `senators_df.TwitterHandle` column
    * `{"$text": {"$search": senator}}` indicates that we’re **using the `text` index** to **`search`** for the value of **`senator`**

In [ ]:
tweet_counts = []

In [ ]:
for senator in senators_df.TwitterHandle: 
    tweet_counts.append(db.tweets.count_documents(
        {"$text": {"$search": senator}}))

<hr style="height:2px; border:none; color:#000; background-color:#000;">

### Show Tweet Counts for Each Senator 
* Create copy of **`DataFrame` `senators_df`** adding a new column of **`tweet_counts`** 
* Display the **top-10 senators by tweet count**

In [ ]:
tweet_counts_df = senators_df.assign(Tweets=tweet_counts)  

In [ ]:
tweet_counts_df

In [ ]:
tweet_counts_df.sort_values(by='Tweets', ascending=False).head(10)

<hr style="height:2px; border:none; color:#000; background-color:#000;">

### Get the State Locations for Plotting Markers (1 of 3)
* Get each **state’s latitude and longitude** coordinates for **plotting on a map**
* **`state_codes.py`** contains a dictionary that maps **two-letter state codes** to **full state names**
    * Used with **`geopy`** to look up the location of each state

In [ ]:
#from geopy import ArcGIS
from geopy import ArcGIS

In [ ]:
import time

In [ ]:
from state_codes import state_codes

* Get the **`geocoder` object** to **translate location names** into **`Location` objects**

In [ ]:
#geo = OpenMapQuest(api_key=keys.mapquest_key) 
geo = ArcGIS() 

<hr style="height:2px; border:none; color:#000; background-
color:#000;">

### Get the State Locations for Plotting Markers (2 of 3)
* Get and sort the unique state names

In [ ]:
states = tweet_counts_df.State.unique()  # get unique state names

In [ ]:
states = sorted(states) # states.sort() 

<hr style="height:2px; border:none; color:#000; background-color:#000;">

### Get the State Locations for Plotting Markers (3 of 3)
* Look up **each state’s location**
* Call `geocode` with state name followed by `', USA'` 
    * Ensures that we get United States locations

In [ ]:
locations = []

In [ ]:
from IPython.display import clear_output

for state in states:
    processed = False
    delay = .1 
    while not processed:
        try: 
            locations.append(geo.geocode(state_codes[state] + ', USA'))
            clear_output()  # clear cell's current output before showing next one
            print(locations[-1])  
            processed = True
        except:  # timed out, so wait before trying again
            print('OpenMapQuest service timed out. Waiting.')
            time.sleep(delay)
            delay += .1

<hr style="height:2px; border:none; color:#000; background-color:#000;">

### Grouping the Tweet Counts by State 
* **Tweet total** for a states' two senators is used to **color the map**
    * **Darker colors** represent **higher tweet counts**
* **`DataFrame` method `groupby`** to group the senators by state 
    * **`as_index=False`**&mdash;state codes should be a column in returned **`GroupBy`** object, rather than indices for the object's rows
* **`GroupBy`** object's **`sum`** method totals the numeric data by state

In [ ]:
tweet_counts_df[['State', 'Tweets']]

In [ ]:
tweets_counts_by_state = tweet_counts_df[['State', 'Tweets']].groupby(
    'State', as_index=False).sum(numeric_only=True)

In [ ]:
tweets_counts_by_state

<hr style="height:2px; border:none; color:#000; background-color:#000;">

### Creating the Map 

In [ ]:
import folium

In [ ]:
usmap = folium.Map(location=[39.8283, -98.5795], 
    zoom_start=4, detect_retina=True)  

######

<!-- #usmap = folium.Map(location=[39.8283, -98.5795], 
    #tiles=tile_url,
    #attr='Map tiles by Stamen Design, under CC BY 4.0. Data by OpenStreetMap, under ODbL.',
#    zoom_start=4, detect_retina=True)  -->

<!--usmap = folium.Map(location=[39.8283, -98.5795], 
                   zoom_start=4, detect_retina=True,
                   tiles='Stamen Toner')-->

In [ ]:
#base_tile_url = 'https://tiles.stadiamaps.com/tiles/stamen_toner/{z}/{x}/{y}@2x.png'
#tile_url = f'{base_tile_url}?api_key="{keys.stadia_key}")'

<hr style="height:2px; border:none; color:#000; background-color:#000;">

### Creating a Choropleth to Color the Map
* A **choropleth** shades areas in a map using magnitudes of numerical values to determine color
* For a **detailed description of the arguments** below, see 
    * [**Python Fundamentals LiveLessons videos**](https://learning.oreilly.com/videos/python-fundamentals/9780135917411/9780135917411-PFLL_Lesson16_23) 
    * [**Python for Programmers, Section 16.4.2** (under the heading "Creating a Choropleth to Color the Map"](https://learning.oreilly.com/library/view/python-for-programmers/9780135231364/ch16.xhtml#ch16lev2sec15)

In [ ]:
choropleth = folium.Choropleth(
    geo_data='us-states.json',
    name='choropleth',
    data=tweets_counts_by_state,
    columns=['State', 'Tweets'],
    key_on='feature.id',
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Tweets by State'
).add_to(usmap)

layer = folium.LayerControl().add_to(usmap)

<hr style="height:2px; border:none; color:#000; background-color:#000;">

### Creating the Map Markers for Each State (1 of 1)
* Sort senators in **descending order** by **tweet count**
* **`groupby`** maintains **original row order** in each group
* **`index`** &mdash; used to look up each state’s location in **`locations` list**
* **name** &mdash; two-letter **state code**
* **`group`** &mdash; collection of a **state's two senators**

In [ ]:
sorted_df = tweet_counts_df.sort_values(by='Tweets', ascending=False)

for index, (name, group) in enumerate(sorted_df.groupby('State')):
    strings = [state_codes[name]]  # used to assemble popup text

    for s in group.itertuples():
        strings.append(f'{s.Name} ({s.Party}); Tweets: {s.Tweets}')
        
    text = '<br>'.join(strings)  
    popup = folium.Popup(text, max_width=200)
    marker = folium.Marker(
        (locations[index].latitude, locations[index].longitude), 
        popup=popup)
    marker.add_to(usmap) 

<hr style="height:2px; border:none; color:#000; background-color:#000;">

### Saving and Displaying the Map 
* May need to execute from the command line
> `jupyter trust Ch16_Part1.ipynb`

In [ ]:
usmap.save('SenatorsTweets.html')

In [ ]:
usmap 
#from IPython.display import IFrame
#IFrame(src="./SenatorsTweets.html", width=800, height=450)

<hr style="height:2px; border:none; color:#000; background-color:#000;">

# More Info 
* See Lesson 16 in [**Python Fundamentals LiveLessons** here on O'Reilly Online Learning](https://learning.oreilly.com/videos/python-fundamentals/9780135917411)
* See Chapter 16 in [**Python for Programmers** on O'Reilly Online Learning](https://learning.oreilly.com/library/view/python-for-programmers/9780135231364/)
* See Chapter 17 in [**Intro Python for Computer Science and Data Science** on O'Reilly Online Learning](https://learning.oreilly.com/library/view/intro-to-python/9780135404799/)
* Interested in a print book? Check out:

| Python for Programmers<br>(640-page professional book) | Intro to Python for Computer<br>Science and Data Science<br>(880-page college textbook)
| :------ | :------
| <a href="https://amzn.to/2VvdnxE"><img alt="Python for Programmers cover" src="../images/PyFPCover.png" width="150" border="1"/></a> | <a href="https://amzn.to/2LiDCmt"><img alt="Intro to Python for Computer Science and Data Science: Learning to Program with AI, Big Data and the Cloud" src="../images/IntroToPythonCover.png" width="159" border="1"></a>

>Please **do not** purchase both books&mdash;_Python for Programmers_ is a subset of _Intro to Python for Computer Science and Data Science_

<hr style="height:2px; border:none; color:#000; background-color:#000;">

------
&copy; 2026 by Deitel & Associates, Inc. All Rights Reserved. The content in this notebook is based on the textbook [**Intro Python for Computer Science and Data Science**](https://amzn.to/2YU0QTJ) and our professional book [**Python for Programmers**](https://amzn.to/2VvdnxE) — Please do not purchase both. The professional book is a subset of the textbook.

DISCLAIMER: The authors and publisher of this book have used their 
best efforts in preparing the book. These efforts include the 
development, research, and testing of the theories and programs 
to determine their effectiveness. The authors and publisher make 
no warranty of any kind, expressed or implied, with regard to these 
programs or to the documentation contained in these books. The authors 
and publisher shall not be liable in any event for incidental or 
consequential damages in connection with, or arising out of, the 
furnishing, performance, or use of these programs.                  